# S01 — Your first email spam classifier 📧

**Goal:** Learn supervised classification with an understandable end-to-end example: labeled emails → features → training → test → predict spam or ham.

**Spam** means unwanted or deceptive promotional messages; **ham** means ordinary legitimate email. We provide each email's **subject**, **body**, **sender**, and correct **label**. After training, the model receives the first three fields and predicts a label.

This is a **small, synthetic classroom demonstration**, not a reliable production spam filter. The dataset has just 24 fabricated messages, so accuracy will be unstable. No real email addresses or private messages are used.

**Connections to LO1:** the known labels make this supervised learning. Compare with unsupervised clustering (no labels) and reinforcement learning (actions and rewards over time).

## Step 1 — Import the libraries

- **pandas**: read and inspect our email table.
- **train_test_split**: reserve 20% for an initial check.
- **TfidfVectorizer**: turn words into numeric features and downweight very common words.
- **LogisticRegression**: learn weights for binary classification (despite its name, this is a **classification** model, not ordinary linear regression).
- **Pipeline**: combine vectorization and training so we apply exactly the same transformation to new emails.
- **accuracy_score** and **confusion_matrix**: inspect predictions, with caution.

Google Colab normally includes these packages. No external service, GPU or API key is required.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
print("Libraries loaded successfully")

## Step 2 — Load a tiny labeled CSV

This notebook can run independently in Colab. The same 24 examples are also saved in [email-spam-example.csv](email-spam-example.csv) in the repository. Below we embed the identical data so there is **no upload step**. For a real CSV you could instead use `pd.read_csv("email-spam-example.csv")` after uploading it to Colab.

Each row represents one email. `label` is what we want the model to learn, so we **never include it in the input features**.

In [ ]:
from io import StringIO
csv_text = """subject,body,sender,label
Urgent verify password,Click now to confirm your account,security@account-alert.example,spam
Claim your prize,You won a voucher claim today,reward@prize-mail.example,spam
Limited offer,Buy now for an exclusive discount,deals@flash-offer.example,spam
Account suspended,Urgent action required verify login,notice@secure-update.example,spam
Free money,Get cash fast click this link,promo@cash-deal.example,spam
Winner notification,Claim your reward immediately,winner@lucky-promo.example,spam
Reset password now,Unusual activity click here to unlock,help@urgent-access.example,spam
Exclusive bonus,Congratulations collect free gift,bonus@special-promo.example,spam
Act immediately,Your package is blocked pay a fee,delivery@fee-claim.example,spam
Huge discount,Limited time click to purchase,sale@deal-zone.example,spam
Important alert,Verify payment details urgently,billing@verify-now.example,spam
Special investment,Guaranteed return sign up today,offers@easy-profit.example,spam
Team meeting,Meeting tomorrow at ten,colleague@office.example,ham
Project update,I uploaded the latest document,manager@company.example,ham
Lunch plans,Would you like lunch on Friday,friend@personal.example,ham
Course material,Here are the lecture slides,lecturer@university.example,ham
Invoice received,Your invoice is attached for review,accounts@vendor.example,ham
Appointment reminder,See you at the clinic on Monday,reception@clinic.example,ham
Weekend plan,We can meet at the park,family@home.example,ham
Code review,Please review the changes tomorrow,developer@company.example,ham
Monthly report,Attached is our sales summary,analyst@office.example,ham
Class schedule,The next lesson starts on Tuesday,admin@university.example,ham
Document shared,I shared the notes from yesterday,peer@company.example,ham
Dinner invitation,Are you free for dinner tonight,friend2@personal.example,ham
"""
emails = pd.read_csv(StringIO(csv_text))
print("Dataset shape:", emails.shape)
print(emails["label"].value_counts())
display(emails.head())

## Step 3 — Turn the three input fields into one text string

The algorithm needs a fixed input representation. For simplicity, we concatenate **subject**, **body** and **sender** into one text field. The model sees all three. This technique is basic: it does not fully capture sender reputation, header authentication, or spoofing.

Example: `subject: Urgent verify password body: Click now ... sender: security@...`

This also illustrates the difference between **X** (the inputs) and **y** (the known correct label).

In [ ]:
emails["text"] = (
    "subject: " + emails["subject"].fillna("") +
    " body: " + emails["body"].fillna("") +
    " sender: " + emails["sender"].fillna("")
)
X = emails["text"]
y = emails["label"]
print("An input example:\n", X.iloc[0])
print("Known correct label:", y.iloc[0])

## Step 4 — Split into training (80%) and testing (20%)

The model learns using **training** emails only. Testing emails are held back until after fitting, to check whether predictions generalise to previously unseen examples.

With 24 records, scikit-learn allocates **19 to train and 5 to test** (approximately 80/20). `random_state=42` makes the split repeatable. `stratify=y` keeps both classes represented proportionally.

**Important:** Five test messages do **not** establish dependable real-world performance. Do not repeatedly change the model because it scored well or badly on these five records.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training:", len(X_train), "Testing:", len(X_test))
print("Train labels:\n", y_train.value_counts())
print("Test labels:\n", y_test.value_counts())

## Step 5 — Convert words to numbers and train the model

Computers cannot directly multiply strings by learned numerical weights. `TfidfVectorizer` converts training text into a matrix where columns correspond to observed word patterns and cells contain numerical weights. It learns its vocabulary and weighting **only from the training subset**.

We use **logistic regression**, a classic supervised **binary classifier**. It learns a weighted combination of text features and converts the result into an estimated probability using a logistic (sigmoid) function. For the two classes it can choose the one with the higher estimated probability.

A `Pipeline` prevents us from accidentally fitting a new vocabulary on test emails. Do not confuse logistic regression with **linear regression**, which predicts a continuous numeric value and is unsuitable here without an additional, unjustified thresholding design.

In [ ]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])
model.fit(X_train, y_train)
print("Training finished.")
print("Learned vocabulary size:", len(model.named_steps["tfidf"].vocabulary_))
print("Classes:", list(model.classes_))

## Step 6 — Predict on emails not used for training

`predict` returns the predicted label for each test message. We compare predictions against their known correct labels. Accuracy is the proportion correct; a confusion matrix counts different correct/incorrect combinations.

This dataset is intentionally tiny and easy; even a good-looking score should not be mistaken for reliable spam detection. A real system needs large representative samples and checks of **false positives** (legitimate email marked spam) and **false negatives** (spam allowed through).

In [ ]:
predicted = model.predict(X_test)
results = pd.DataFrame({"subject_body_sender": X_test.values, "actual": y_test.values, "predicted": predicted})
display(results)
print("Accuracy on five test emails:", round(accuracy_score(y_test, predicted), 3))
print("Confusion matrix, label order [ham, spam]:\n",
      confusion_matrix(y_test, predicted, labels=["ham", "spam"]))

## Step 7 — Give the model a brand-new email

The function takes three parameters — `subject`, `body`, and `sender` — and returns **spam or ham** plus the model's estimated spam probability.

**Interpret with care:** this number is not guaranteed to be a calibrated probability with a training set this small. A real mail service also uses authentication, links, reputation, and policy rules.

In [ ]:
def predict_email(subject, body, sender):
    text = f"subject: {subject} body: {body} sender: {sender}"
    label = model.predict([text])[0]
    probabilities = model.predict_proba([text])[0]
    spam_index = list(model.classes_).index("spam")
    return {"prediction": label, "spam_probability_estimate": round(float(probabilities[spam_index]), 3)}

example = predict_email(
    subject="Urgent: claim your free prize",
    body="Click now to verify your account and collect the reward",
    sender="reward@special-promo.example"
)
print(example)

## Step 8 — Try a normal message, then experiment

Modify subject, body and sender. Notice that words learned from only 24 examples cannot capture every real-world case. In particular, legitimate emails sometimes mention passwords or urgent issues. A simple keyword pattern is not proof of fraud.

In [ ]:
print(predict_email(
    subject="Lecture slides",
    body="Hello, I attached the material for tomorrow's class",
    sender="lecturer@university.example"
))

## Student discussion and independent practice

1. Which columns are **features** and which column is the **label**? Why is this supervised learning?
2. Why must the TF-IDF vocabulary be learned only on training emails?
3. What is a false positive in spam filtering? Why might it be costly?
4. Replace the email in Step 7 with your own invented example (no real private emails).
5. Change the split seed in your own copy. Do results vary? Why are five test messages insufficient?
6. In one paragraph, explain the difference between linear regression, logistic regression and unsupervised clustering.

**Extension (optional):** Compare a `DummyClassifier(strategy="most_frequent")` baseline and discuss why accuracy alone can mislead on a heavily imbalanced inbox.

**Takeaway:** We collected labeled observations, represented their text numerically, fitted a supervised classifier and produced predictions for new input. S02 will examine preprocessing and data leakage more rigorously.